# CDO CSV to Parquet

Read the Bureau of Meteorology CDO CSV files, ignore the description lines at the top of each file, combine each station/product prefix into one dataset, convert date/time fields to timestamps, and write parquet files.

In [2]:
from pathlib import Path
import re

import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "Writing parquet requires pyarrow. Install it in your conda env, for example: "
        "conda install -n pyeartools -c conda-forge pyarrow"
    ) from exc

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    REPO_ROOT = NOTEBOOK_DIR.parents[2]
else:
    REPO_ROOT = Path("/home/timekeeper/Documents/Development/BOM-Team")

DATA_DIR = REPO_ROOT / "data" / "CDO"
OUTPUT_DIR = DATA_DIR / "parquet"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFIXES = ["IDCJDW2008", "IDCJDW2101"]
DESCRIPTION_ROWS = 6

DATA_DIR, OUTPUT_DIR

(PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/CDO'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/CDO/parquet'))

In [3]:
def clean_column_name(column: str) -> str:
    """Make BOM column names easier to use while preserving their meaning."""
    column = str(column).strip()
    column = column.replace("\ufffdC", "C")
    column = re.sub(r"\s+", " ", column)
    return column


def parse_station_id(path: Path) -> str | None:
    for line in path.read_text(encoding="latin1").splitlines()[:6]:
        match = re.search(r"\{station\s+([^}]+)\}", line)
        if match:
            return match.group(1)
    return None


def read_cdo_csv(path: Path, prefix: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=DESCRIPTION_ROWS, encoding="latin1")

    unnamed_columns = [column for column in df.columns if str(column).startswith("Unnamed:")]
    df = df.drop(columns=unnamed_columns)
    df.columns = [clean_column_name(column) for column in df.columns]

    df["source_id"] = prefix
    df["source_file"] = path.name
    df["station_id"] = parse_station_id(path)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    gust_time_col = "Time of maximum wind gust"
    if gust_time_col in df.columns:
        gust_time = df[gust_time_col].astype("string").str.strip()
        valid_gust_time = gust_time.notna() & gust_time.str.match(r"^\d{1,2}:\d{2}$", na=False)
        df["Maximum wind gust timestamp"] = pd.NaT
        df.loc[valid_gust_time, "Maximum wind gust timestamp"] = pd.to_datetime(
            df.loc[valid_gust_time, "Date"].dt.strftime("%Y-%m-%d") + " " + gust_time[valid_gust_time],
            errors="coerce",
        )

    return df


def normalize_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    object_columns = df.select_dtypes(include=["object"]).columns
    for column in object_columns:
        df[column] = df[column].astype("string").str.strip().replace("", pd.NA)
    return df


def combine_prefix(prefix: str) -> pd.DataFrame:
    files = sorted(DATA_DIR.glob(f"{prefix}.*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found for {prefix} in {DATA_DIR}")

    frames = [read_cdo_csv(path, prefix) for path in files]
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["Date", "source_file"], ignore_index=True)
    combined = normalize_for_parquet(combined)
    return combined


sample = combine_prefix(PREFIXES[0])
sample.dtypes

/tmp/ipykernel_71424/2308476641.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include=["object"]).columns


Date                                 datetime64[us]
Minimum temperature (°C)                    float64
Maximum temperature (°C)                    float64
Rainfall (mm)                               float64
Evaporation (mm)                            float64
Sunshine (hours)                            float64
Direction of maximum wind gust               string
Speed of maximum wind gust (km/h)           float64
Time of maximum wind gust                    string
9am Temperature (°C)                        float64
9am relative humidity (%)                   float64
9am cloud amount (oktas)                    float64
9am wind direction                           string
9am wind speed (km/h)                        string
9am MSL pressure (hPa)                      float64
3pm Temperature (°C)                        float64
3pm relative humidity (%)                   float64
3pm cloud amount (oktas)                    float64
3pm wind direction                           string
3pm wind spe

In [4]:
written_files = []

for prefix in PREFIXES:
    combined = combine_prefix(prefix)
    output_path = OUTPUT_DIR / f"{prefix}.parquet"
    combined.to_parquet(output_path, index=False, engine="pyarrow")
    written_files.append(
        {
            "prefix": prefix,
            "rows": len(combined),
            "date_min": combined["Date"].min(),
            "date_max": combined["Date"].max(),
            "output_path": str(output_path),
        }
    )

pd.DataFrame(written_files)

/tmp/ipykernel_71424/2308476641.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include=["object"]).columns
/tmp/ipykernel_71424/2308476641.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-

,prefix,rows,date_min,date_max,output_path
0,IDCJDW2008,406,2025-04-01,2026-05-11,/home/timekeeper/Documents/Development/BOM-Tea...
1,IDCJDW2101,406,2025-04-01,2026-05-11,/home/timekeeper/Documents/Development/BOM-Tea...


In [5]:
idcjdw2008 = pd.read_parquet(OUTPUT_DIR / "IDCJDW2008.parquet", engine="pyarrow")
idcjdw2008.head()

,Date,Minimum temperature (°C),Maximum temperature (°C),Rainfall (mm),Evaporation (mm),Sunshine (hours),Direction of maximum wind gust,Speed of maximum wind gust (km/h),Time of maximum wind gust,9am Temperature (°C),...,3pm Temperature (°C),3pm relative humidity (%),3pm cloud amount (oktas),3pm wind direction,3pm wind speed (km/h),3pm MSL pressure (hPa),source_id,source_file,station_id,Maximum wind gust timestamp
0,2025-04-01,16.6,25.0,0.0,NaN,NaN,S,37.0,13:51,19.4,...,22.4,54.0,7.0,S,28,1013.1,IDCJDW2008,IDCJDW2008.202504.csv,066137,2025-04-01 13:51:00
1,2025-04-02,13.7,25.3,0.0,NaN,NaN,ESE,31.0,16:30,15.3,...,24.8,33.0,NaN,ESE,6,1007.9,IDCJDW2008,IDCJDW2008.202504.csv,066137,2025-04-02 16:30:00
2,2025-04-03,12.1,27.1,0.0,NaN,NaN,SW,33.0,11:59,16.3,...,26.9,30.0,NaN,SSW,20,1009.3,IDCJDW2008,IDCJDW2008.202504.csv,066137,2025-04-03 11:59:00
3,2025-04-04,13.3,26.0,0.0,NaN,NaN,ESE,35.0,15:01,17.8,...,24.5,55.0,8.0,SE,19,1014.2,IDCJDW2008,IDCJDW2008.202504.csv,066137,2025-04-04 15:01:00
4,2025-04-05,13.9,26.9,0.0,NaN,NaN,SE,39.0,14:30,18.6,...,24.4,34.0,NaN,SE,26,1014.5,IDCJDW2008,IDCJDW2008.202504.csv,066137,2025-04-05 14:30:00
